In [ ]:
from pathlib import Path
import re
import yaml
import os
from openpyxl import load_workbook

In [ ]:
!date
jupyter_dir = os.getcwd()
print(f"DIR: {jupyter_dir}")

In [ ]:
#!date
#excel_file = Path("./Subscriber.xlsx")
#out_yaml = Path("./output2/output.yaml")

In [ ]:
!date
from pathlib import Path
import re
import yaml
from openpyxl import load_workbook

# ====== Excel layout ======
SHEETS = ["SourceAddress", "BlackList", "WhiteList", "CategoryList"]

ROW_IDENTIFIER = 4
ROW_DATA_START = 5
COL_START = 3

# ====== BIG-IP fixed rules ======
MASK_FIXED = "255.255.255.255"
VS_RULE = "/Common/Shared/url_filter_irule"
TMC_DESTINATION_ADDRESS_INLINE = "192.168.23.12"

TCP_PROFILES = [
    "/Common/Shared/{id}_Profile",
    "/Common/Shared/{id}_stats",
    "/Common/tcp",
]

UDP_PROFILES = [
    "/Common/Shared/{id}_Profile",
    "/Common/Shared/{id}_stats",
    "/Common/Shared/udp_gtm_dns",
]

split_pattern = re.compile(r"[,\n\r]+")


def split_values(v):
    if v is None:
        return []
    s = str(v).strip()
    if not s:
        return []
    return [x.strip() for x in split_pattern.split(s) if x.strip()]


def read_identifier_columns(ws):
    col_to_id = {}
    for col in range(COL_START, ws.max_column + 1):
        v = ws.cell(row=ROW_IDENTIFIER, column=col).value
        if v is None:
            continue
        ident = str(v).strip()
        if ident:
            col_to_id[col] = ident
    return col_to_id


def read_sheet_values_by_identifier(ws):
    col_to_id = read_identifier_columns(ws)
    result = {ident: [] for ident in col_to_id.values()}

    for col, ident in col_to_id.items():
        values = []
        for row in range(ROW_DATA_START, ws.max_row + 1):
            cell_v = ws.cell(row=row, column=col).value
            values.extend(split_values(cell_v))

        values = [v for v in values if v and v.strip()]
        result[ident] = sorted(set(values))

    return result


def build_virtual_servers_expected(identifier: str):
    return {
        "TCP": {
            "name": f"/Common/Shared/{identifier}-VIP_P_TCP",
            "ip_protocol": "tcp",
            "mask": MASK_FIXED,
            "profiles": sorted({p.format(id=identifier) for p in TCP_PROFILES}),
            "rules": [VS_RULE],
            "traffic_matching_criteria": f"/Common/Shared/{identifier}-VIP_P_TCP_VS_TMC_OBJ",
        },
        "UDP": {
            "name": f"/Common/Shared/{identifier}-VIP_P_UDP",
            "ip_protocol": "udp",
            "mask": MASK_FIXED,
            "profiles": sorted({p.format(id=identifier) for p in UDP_PROFILES}),
            "rules": [VS_RULE],
            "traffic_matching_criteria": f"/Common/Shared/{identifier}-VIP_P_UDP_VS_TMC_OBJ",
        },
    }


def build_traffic_matching_criteria_expected(identifier: str):
    return {
        "TCP": {
            "name": f"/Common/Shared/{identifier}-VIP_P_TCP_VS_TMC_OBJ",
            "destination_address_inline": TMC_DESTINATION_ADDRESS_INLINE,
            "source_address_list": f"/Common/Shared/{identifier}_addresslist",
        },
        "UDP": {
            "name": f"/Common/Shared/{identifier}-VIP_P_UDP_VS_TMC_OBJ",
            "destination_address_inline": TMC_DESTINATION_ADDRESS_INLINE,
            "source_address_list": f"/Common/Shared/{identifier}_addresslist",
        },
    }


def excel_to_expected(excel_path: str | Path):
    excel_path = Path(excel_path)
    wb = load_workbook(excel_path, data_only=True)

    sheet_data = {}
    for sheet in SHEETS:
        if sheet not in wb.sheetnames:
            raise ValueError(f"Sheet not found: {sheet}. Available: {wb.sheetnames}")
        ws = wb[sheet]
        sheet_data[sheet] = read_sheet_values_by_identifier(ws)

    identifiers = set()
    for d in sheet_data.values():
        identifiers |= set(d.keys())

    expected = {}
    for ident in sorted(identifiers):
        expected[ident] = {

            # ここを変更
            "source_address_list": sheet_data["SourceAddress"].get(ident, []),

            "data_groups": {
                "blacklist": sheet_data["BlackList"].get(ident, []),
                "whitelist": sheet_data["WhiteList"].get(ident, []),
                "category": sheet_data["CategoryList"].get(ident, []),
            },

            "virtual_servers": build_virtual_servers_expected(ident),

            "traffic_matching_criteria": build_traffic_matching_criteria_expected(ident),
        }

    return expected


# ====== Run ======
excel_file = Path("./Subscriber.xlsx")
out_yaml = Path("./output2/output.yaml")

expected = excel_to_expected(excel_file)

out_yaml.parent.mkdir(parents=True, exist_ok=True)

with open(out_yaml, "w", encoding="utf-8") as f:
    yaml.dump(expected, f, allow_unicode=True, sort_keys=False)

print(f"Exported: {out_yaml} (identifiers={len(expected)})")


In [ ]:
#expected = excel_to_expected(excel_file)
#
## 出力ディレクトリが無ければ作成
#out_yaml.parent.mkdir(parents=True, exist_ok=True)
#
## YAML出力
#with open(out_yaml, "w", encoding="utf-8") as f:
#    yaml.dump(expected, f, allow_unicode=True, sort_keys=False)
#
#print(f"Exported: {out_yaml} (identifiers={len(expected)})")